# NB5c conformal (LightGBM) — calibrated intervals for LightGBM's 5-seed ensemble (validation units only)

D06, D35, D46, D64. The author decided on 21 Nov 2026 that LightGBM enters the test as its 5-seed ensemble (`LGB_seeds`, D37). Its conformal layer is therefore rebuilt from that ensemble, with **the rules of D35, unchanged**, exactly as for PVSetFormer (`NB5c_conformal`). Reads `results/nb5c_variants/LGB_seeds/` and `results/nb5c_variants/LGB_seeds_cal/` (`tools/lgb_seeds_cal.py`); writes `results/nb5c_conformal_lgb/`. No test event is used.

**Rules (D35, fixed before any number below):**
1. *Calibration set*, per event, origin and horizon: pseudo cold starts of the event's donors (clean-metadata donors for a G1/G2 target, all donors for a G0 target). Their index is forecast by the early-stopped models on the last 20% of the training pairs (out of sample for those models, targets before the origin), here the 5-seed ensemble of these predictions, built like the forecasts (mean of the sorted quantiles, sorted again), and composed with each donor's leave-one-out prior and uncertainty set (200 members).
2. *Conformity*: CQR (Romano et al. 2019) per horizon and per central interval (80, 60, 40 and 20%), normalised by G̃(t + h). The correction is the ⌈(1 − α)(n + 1)⌉-th smallest score; under 50 scores, none.
3. *Use*: applied to LightGBM's 9 power quantiles; the median and the point forecast are unchanged.
4. *Keep rule*: the layer is kept if, on the validation units, it brings the median PICP80 at 1 h closer to 0.80 than the raw forecast and does not raise the median IS80 at 1 h.

In [ ]:
import os, sys, time, json, hashlib
from pathlib import Path
import numpy as np, pandas as pd, yaml
ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == 'notebooks' else ROOT
sys.path.insert(0, str(ROOT / 'src')); sys.path.insert(0, str(ROOT / 'tools'))
from pvcold import config as C, audit as A, fleet as F, forecast as FC
import r1_score as R                      # the scoring shared by rule R1 (D37)
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
t0 = time.time()
VDIR, CDIR = C.RESULTS / 'nb5c_variants' / 'LGB_seeds', C.RESULTS / 'nb5c_variants' / 'LGB_seeds_cal'
nb5 = json.loads((C.RESULTS / 'nb5a_dev' / 'summary.json').read_text())
ls_bytes = (VDIR / 'summary.json').read_bytes(); ls = json.loads(ls_bytes)
lc = json.loads((CDIR / 'summary.json').read_text())
P2A_SHA = hashlib.sha256((C.RESULTS / 'nb5a_dev' / 'lgb_config_part2a.yaml').read_bytes()).hexdigest()
assert P2A_SHA == nb5['lgb_config_sha256'] == ls['lgb_config_sha256'] == lc['lgb_config_sha256'], 'configuration part 2a changed'
assert hashlib.sha256(ls_bytes).hexdigest() == lc['lgb_seeds_summary_sha256'], 'LGB_seeds changed after its calibration run'
SEEDS = ls['seeds']
assert lc['seeds'] == SEEDS and lc['forecasts_max_abs_diff'] < 1e-6
assert lc['calibration_keys_with_all_seeds'] == lc['calibration_keys'], 'a calibration key lacks a seed'
ctx = R.load_context()
p, E, U, geo15, gains3 = ctx['p'], ctx['E'], ctx['U'], ctx['geo15'], ctx['gains3']
PRIOR, ALT, VAL, fc5 = ctx['PRIOR'], ctx['ALT'], ctx['VAL'], ctx['fc5']
QC = R.QC
lv, gl = FC.member_levels()
def ensemble(x):
    x = x[x.seed != 0]
    n = x.groupby(['event', 'origin', 'h', 't'])['seed'].nunique()
    assert sorted(x.seed.unique().tolist()) == sorted(SEEDS) and (n == len(SEEDS)).all(), 'seeds differ'
    e = R.with_ensemble(x.drop(columns='model'))
    return e[e.seed == 0].drop(columns='seed').reset_index(drop=True)
fc = ensemble(pd.read_parquet(VDIR / 'forecasts.parquet'))
cal = ensemble(pd.read_parquet(CDIR / 'calibration.parquet'))
print('LightGBM 5-seed ensemble | seeds', SEEDS, '| prior', PRIOR)
print('ensemble rows: forecasts %d, calibration %d' % (len(fc), len(cal)))

## Conformity scores of the donors' pseudo cold starts, and the corrections

In [ ]:
Tcache, crow, ev_gain = {}, [], {}
for v in VAL:
    a = F.event_arrays(v, p)
    g = gains3[gains3.event == v].set_index('donor')['gain'].reindex(a['dn'])
    pr, T_don, r_don, meta_ok = F.event_priors(a, g, U, ALT, Tcache)
    variant = PRIOR if pr['target_in_prior_test'] else 'G0'
    B = np.asarray(pr['B_' + variant]) if variant != 'G0' else g.dropna().to_numpy()
    ev_gain[v] = FC.gain_members(pr[variant], B / np.median(B), gl)
    pp = FC.pseudo_priors(g, r_don, T_don, meta_ok, variant)
    for (o, h), x in cal[cal.event == v].groupby(['origin', 'h']):
        tgt = x['t'] + pd.Timedelta(minutes=15 * h)
        gtt = geo15['Gt'].reindex(tgt).to_numpy(); Qi = x[QC].to_numpy(dtype=float)
        Es = []
        for d, (g0, Brel) in pp.items():
            yd = p['y15'][d].reindex(tgt).to_numpy(); ok = np.isfinite(yd)
            Y = FC.power_members(Qi[ok], gtt[ok], FC.gain_members(g0, Brel, gl), lv)
            Es.append(FC.conformity(FC.power_quantiles(Y), yd[ok], gtt[ok]))
        Eall = np.vstack(Es)
        corr = FC.conformal_correction(Eall)
        crow.append({'event': v, 'origin': o, 'h': h, 'variant': variant, 'n_donors': len(pp), 'n_scores': len(Eall),
                     'cal_days': int(x['t'].dt.normalize().nunique()),
                     **{f'raw_cover{n}': float(np.mean(Eall[:, j] <= 0)) for j, n in enumerate(FC.PAIR_NAMES)},
                     **{f'corr{n}': float(corr[j]) for j, n in enumerate(FC.PAIR_NAMES)}})
corrs = pd.DataFrame(crow)
print('variants of the prior:', corrs.groupby('variant').event.nunique().to_dict())
print('raw coverage of the pseudo cold starts and corrections (median over units and origins):')
print(corrs.groupby('h')[[c for c in corrs.columns if c.startswith(('raw_cover', 'corr', 'n_scores', 'cal_days'))]].median().round(3).to_string())
print('groups with fewer than %d scores (no correction): %d of %d' % (FC.MIN_CAL, int((corrs.n_scores < FC.MIN_CAL).sum()), len(corrs)))

## Validation units (true cold starts): raw against conformal intervals
Same scored slots as NB5a and rule R1 (every NB5a model has a forecast and the observation exists).

In [ ]:
cidx = corrs.set_index(['event', 'origin', 'h'])
srows = []
for v in VAL:
    assert E.loc[v, 'set'] == 'validation'      # target power is read for validation units only
    f5 = fc5[fc5.event == v].copy()
    f5['target'] = f5['t'] + pd.to_timedelta(15 * f5['h'], unit='min')
    f5['y'] = p['y15'][E.loc[v, 'units']].reindex(f5['target']).to_numpy()
    good = f5[QC].notna().all(axis=1) & np.isfinite(f5['y'])
    cnt = f5[good].groupby(['h', 't'])['model'].nunique()
    keys = cnt[cnt == f5.model.nunique()].index
    f = fc[fc.event == v].set_index(['h', 't']).loc[lambda x: x.index.isin(keys)].reset_index()
    f['target'] = f['t'] + pd.to_timedelta(15 * f['h'], unit='min')
    f['y'] = p['y15'][E.loc[v, 'units']].reindex(f['target']).to_numpy()
    f['gt'] = geo15['Gt'].reindex(f['target']).to_numpy()
    for h, xh in f.groupby('h'):
        assert len(xh) == int((keys.get_level_values(0) == h).sum()), 'missing scored slots'
        parts = {'raw': [], 'conformal': []}; ys, tm = [], []
        for o, x in xh.groupby('origin'):
            gt_ = x['gt'].to_numpy()
            Qp = FC.power_quantiles(FC.power_members(x[QC].to_numpy(dtype=float), gt_, ev_gain[v], lv))
            corr = cidx.loc[(v, o, h), [f'corr{n}' for n in FC.PAIR_NAMES]].to_numpy(dtype=float)
            parts['raw'].append(Qp); parts['conformal'].append(FC.apply_conformal(Qp, gt_, corr))
            ys.append(x['y'].to_numpy()); tm.append((x['target'].dt.hour * 60 + x['target'].dt.minute).to_numpy())
        y_, tm_ = np.concatenate(ys), np.concatenate(tm)
        for kind, Qs in parts.items():
            Qa = np.vstack(Qs)
            for win, msk in [('full', np.ones(len(y_), bool)), ('common', tm_ >= FC.COMMON_WINDOW_MIN)]:
                srows.append({'event': v, 'h': h, 'kind': kind, 'window': win, **FC.quantile_scores(Qa[msk], y_[msk], lv)})
vs = pd.DataFrame(srows)
r1sc = pd.read_csv(VDIR / 'scores.csv').query('seed == 0').set_index(['event', 'h'])['PICP80']
chk = vs[(vs.kind == 'raw') & (vs.window == 'full')].set_index(['event', 'h'])['cover80']
CHK = float((chk - r1sc.reindex(chk.index)).abs().max())
print('raw PICP80 equals the R1 scores of the ensemble: max |diff| = %.1e' % CHK)
assert CHK < 1e-9
MET = ['cover80', 'cover60', 'cover40', 'cover20', 'PINAW', 'IS80', 'CRPS_q']
vm = vs.groupby(['window', 'kind', 'h'])[MET].median().reset_index()
for met in MET:
    print(f'median {met}, full window:'); print(vm[vm.window == 'full'].pivot(index='kind', columns='h', values=met).round(4).to_string())

## Keep rule (rule 4), and the other layers for reference
NB5a's single-seed LightGBM with its own layer (`results/nb5a_conformal/`) and PVSetFormer with its layer (`results/nb5c_conformal/`), shown for reference only.

In [ ]:
r1h = vm[(vm.window == 'full') & (vm.h == C.PRIMARY_HORIZON)].set_index('kind')
closer = abs(r1h.loc['conformal', 'cover80'] - 0.8) < abs(r1h.loc['raw', 'cover80'] - 0.8)
not_worse = r1h.loc['conformal', 'IS80'] <= r1h.loc['raw', 'IS80']
KEEP = bool(closer and not_worse)
won = vs[(vs.window == 'full') & (vs.h == C.PRIMARY_HORIZON)].pivot(index='event', columns='kind', values='IS80')
print('at 1 h: PICP80 raw %.3f -> conformal %.3f | IS80 raw %.4f -> conformal %.4f | units with lower IS80: %d of %d'
      % (r1h.loc['raw', 'cover80'], r1h.loc['conformal', 'cover80'], r1h.loc['raw', 'IS80'], r1h.loc['conformal', 'IS80'],
         int((won['conformal'] < won['raw']).sum()), len(won)))
print('KEEP the conformal layer for LightGBM (5 seeds):', KEEP)
lgb1 = pd.read_csv(C.RESULTS / 'nb5a_conformal' / 'val_conformal_medians.csv').query("window == 'full'")
pvsf = pd.read_csv(C.RESULTS / 'nb5c_conformal' / 'val_conformal_medians.csv').query("window == 'full'")
ref = pd.concat({'LightGBM, 5 seeds': vm[vm.window == 'full'].set_index(['kind', 'h'])[['cover80', 'IS80', 'CRPS_q']],
                 'LightGBM (NB5a, 1 seed)': lgb1.set_index(['kind', 'h'])[['cover80', 'IS80', 'CRPS_q']],
                 'PVSetFormer (NB5c)': pvsf.set_index(['kind', 'h'])[['cover80', 'IS80', 'CRPS_q']]}, names=['model'])
print(ref.unstack('h').round(4).to_string())

## Draft figure: reliability of the central intervals

In [ ]:
import matplotlib.pyplot as plt
A._style()
f8, ax = plt.subplots(1, 2, figsize=(7.2, 2.8))
nom = np.array([0.2, 0.4, 0.6, 0.8])
for kind, c, mk in [('raw', '#b03a2e', 'o'), ('conformal', '#1f4e79', 's')]:
    r = vm[(vm.window == 'full') & (vm.kind == kind) & (vm.h == C.PRIMARY_HORIZON)].iloc[0]
    ax[0].plot(nom, [r['cover20'], r['cover40'], r['cover60'], r['cover80']], marker=mk, ms=3.5, color=c, lw=1, label=kind)
    z = vm[(vm.window == 'full') & (vm.kind == kind)].set_index('h').loc[FC.H, 'cover80']
    ax[1].plot(np.array(FC.H) * 0.25, z, marker=mk, ms=3.5, color=c, lw=1, label=kind)
ax[0].plot([0, 1], [0, 1], color='0.6', lw=0.7); ax[0].set_xlim(0.1, 0.9); ax[0].set_ylim(0.1, 0.9)
ax[0].set_xlabel('Nominal coverage'); ax[0].set_ylabel('Observed coverage (1 h)'); ax[0].legend(frameon=False)
ax[1].axhspan(0.75, 0.85, color='0.9', lw=0); ax[1].axhline(0.8, color='0.6', lw=0.7)
ax[1].set_xscale('log'); ax[1].set_xticks(np.array(FC.H) * 0.25); ax[1].set_xticklabels(['0.25', '0.5', '1', '2', '4', '6'])
ax[1].set_xlabel('Horizon (h)'); ax[1].set_ylabel('PICP of the 80% interval'); ax[1].set_ylim(0.5, 0.95)
ax[0].set_title('LightGBM (5 seeds), median over the 10 validation units', fontsize=7); f8.tight_layout()

## Outputs (all-or-nothing); LightGBM's 5-seed conformal settings join the frozen part 2

In [ ]:
conf_cfg = {'part': '2b', 'model': 'LightGBM, 5-seed ensemble', 'lgb_config': P2A_SHA,
            'seeds': SEEDS, 'method': 'CQR (Romano et al. 2019), split conformal',
            'calibration_set': 'pseudo cold starts of the event donors; 5-seed ensemble of the early-stopped LightGBM predictions on the last 20% of the training pairs',
            'donors': 'clean metadata (G1/G2 target) or all (G0 target); leave-one-out prior and uncertainty set',
            'intervals': FC.PAIR_NAMES, 'normalisation': 'Gt(t + h)', 'per': 'event, origin, horizon',
            'min_scores': FC.MIN_CAL, 'power_bounds': list(FC.POWER_BOUNDS), 'median_and_point': 'unchanged',
            'keep_rule': 'median PICP80 at 1 h closer to 0.80 and median IS80 at 1 h not higher (validation units)',
            'kept': KEEP}
CONF_TEXT = yaml.safe_dump(conf_cfg, sort_keys=True)
CONF_HASH = hashlib.sha256(CONF_TEXT.encode()).hexdigest()
full = vm[vm.window == 'full']
summary = {'notebook': 'NB5c_conformal_lgb', 'decision': 'D64', 'environment': A.versions(), 'runtime_s': round(time.time() - t0, 1),
           'lgb_config_sha256': P2A_SHA, 'lgb_seeds_summary_sha256': lc['lgb_seeds_summary_sha256'], 'seeds': SEEDS,
           'conformal_config_sha256': CONF_HASH, 'kept': KEEP,
           'raw_picp80_check_max_abs_diff': CHK, 'groups_without_correction': int((corrs.n_scores < FC.MIN_CAL).sum()),
           'n_groups': len(corrs),
           'units_lower_is80_1h': int((won['conformal'] < won['raw']).sum()),
           'median_corrections_by_h': corrs.groupby('h')[[f'corr{n}' for n in FC.PAIR_NAMES]].median().round(4).to_dict('index'),
           'pseudo_raw_cover80_by_h': corrs.groupby('h')['raw_cover80'].median().round(4).to_dict(),
           **{f'{kind}_{met}_by_h': full[full.kind == kind].set_index('h')[met].round(4).to_dict()
              for kind in ['raw', 'conformal'] for met in ['cover80', 'cover60', 'cover40', 'cover20', 'PINAW', 'IS80', 'CRPS_q']}}
out = A.write_outputs('nb5c_conformal_lgb', {'corrections': corrs, 'val_conformal_scores': vs, 'val_conformal_medians': vm},
                      summary, {'fig_conformal_reliability_lgb5': f8}, parquet={'calibration_quantiles_ens': cal},
                      texts={'conformal_config_part2b_lgb5.yaml': CONF_TEXT})
print('conformal config (part 2b, LightGBM 5 seeds) sha256:', CONF_HASH)
print('written to', out, '|', sorted(x.name for x in out.iterdir()))